# Setup and Utils

In [ ]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from functools import partial
from typing import Any, Callable, Literal, TypeAlias

import requests
import random
import einops
import numpy as np
import pandas as pd
import requests
import torch as t
from IPython.display import HTML, IFrame, clear_output, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from sae_lens import SAE
from torch import Tensor, nn
from torch.nn import functional as F
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if t.cuda.is_available() else "mps" if t.backends.mps.is_available() else "cpu"

t.set_grad_enabled(False)
home_dir = "../../"

MODEL_ID = "google/gemma-3-4b-it"
NEURONPEDIA_MODEL = "gemma-3-4b-it"
NEURONPEDIA_API_KEY = os.getenv("NEURONPEDIA_API_KEY", "YOUR_SECRET_TOKEN")
UPSTREAM_SAE_RELEASE = os.getenv("UPSTREAM_SAE_RELEASE", "gemma-scope-2-4b-it-res")
UPSTREAM_SAE_ID_TEMPLATE = "layer_{layer}_width_16k_l0_medium"
UPSTREAM_NEURONPEDIA_LAYER_TEMPLATE = "{layer}-gemmascope-2-res-16k"
AVAILABLE_UPSTREAM_SAE_LAYERS = [9, 17, 22, 29]
DIRECTION_SAE_RELEASE = UPSTREAM_SAE_RELEASE
DIRECTION_SAE_ID_TEMPLATE = UPSTREAM_SAE_ID_TEMPLATE
DIRECTION_LAYER = 17
DIRECTION_FEATURE_IDX = 3290
DIRECTION_NEURONPEDIA_MODEL = NEURONPEDIA_MODEL
DIRECTION_NEURONPEDIA_LAYER = UPSTREAM_NEURONPEDIA_LAYER_TEMPLATE.format(layer=DIRECTION_LAYER)
DIRECTION_FEATURE_LABEL = f"{DIRECTION_NEURONPEDIA_MODEL}/{DIRECTION_NEURONPEDIA_LAYER}/{DIRECTION_FEATURE_IDX}"

def resolve_core_model(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model
    if hasattr(model, "language_model") and hasattr(model.language_model, "layers"):
        return model.language_model
    if hasattr(model, "language_model") and hasattr(model.language_model, "model"):
        inner = model.language_model.model
        if hasattr(inner, "layers"):
            return inner
    raise ValueError("Unsupported model structure: could not find a transformer layer stack.")

## functions
def get_chat_template(prompt, tokenizer):
    prompt = [
        {"role": "user", "content": prompt},
        ]
    prompt_with_template = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    prompt_with_template_tokens = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True)
    
    return prompt_with_template, prompt_with_template_tokens

from IPython.display import IFrame, display

def neuronpedia_layer_name(layer):
    return UPSTREAM_NEURONPEDIA_LAYER_TEMPLATE.format(layer=layer)

def plot_dashboard(feature_idx, layer):
    html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"
    html = html_template.format(NEURONPEDIA_MODEL, neuronpedia_layer_name(layer), feature_idx)
    display(IFrame(html, width=1200, height=600))

def load_tensor(filename, device = device):
    if device == "mps":
        tensor = t.load(filename, map_location="cpu")
        tensor = tensor.to(device, dtype=t.float32)
    else:
        tensor = t.load(filename)
    return tensor

def get_auto_interp(layer, feature_idx):
    url = f"https://www.neuronpedia.org/api/feature/{NEURONPEDIA_MODEL}/{neuronpedia_layer_name(layer)}/{feature_idx}"

    headers = {"X-Api-Key": NEURONPEDIA_API_KEY}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    explanations = response.json().get('explanations', [])
    return explanations[0]["description"] if explanations else None


# Variables to Play Around with

In [ ]:
PROMPT = """How does your company's integration of new AI tools in customer service reflect your brand's commitment to innovation while maintaining the human touch that customers value? I'm interested in how these technologies enhance rather than replace the customer service experience your brand is known for."""
UPSTREAM_LAYER = 17


# Step 1: Load Model and Refusal Direction

In [ ]:
## Load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=t.bfloat16,
    device_map="auto",
)
core_model = resolve_core_model(model)
gemma2 = model

sae_name = UPSTREAM_SAE_RELEASE
sae_id = UPSTREAM_SAE_ID_TEMPLATE.format(layer=UPSTREAM_LAYER)

if UPSTREAM_LAYER not in AVAILABLE_UPSTREAM_SAE_LAYERS:
    raise RuntimeError(
        f"Requested UPSTREAM_LAYER={UPSTREAM_LAYER}, but the installed Gemma Scope 2 4B IT residual subset only has layers "
        f"{AVAILABLE_UPSTREAM_SAE_LAYERS}. Choose one of those layers or switch to the all-layer release."
    )

try:
    gemma2_sae, cfg_dict, sparsity = SAE.from_pretrained(
                release=sae_name,
                sae_id=sae_id,
                device=str(device),
    )
except Exception as e:
    raise RuntimeError(
        f"Could not load upstream SAE {sae_name}/{sae_id}."
    ) from e

## Load steering direction from a Gemma Scope decoder vector shown in Neuronpedia
direction_sae, _, _ = SAE.from_pretrained(
            release=DIRECTION_SAE_RELEASE,
            sae_id=DIRECTION_SAE_ID_TEMPLATE.format(layer=DIRECTION_LAYER),
            device=str(device),
)
refusal_direction = direction_sae.W_dec[DIRECTION_FEATURE_IDX, :].detach().to(device)

refusal_layer = DIRECTION_LAYER
refusal_direction_unit = refusal_direction / t.norm(refusal_direction, p=2)
print(f"Loaded steering direction from {DIRECTION_FEATURE_LABEL}")


# Step 2: Find Promising Latents By Getting Gradient for Refusal Direction

In [ ]:
def get_gradient(model, prompt, upstream_layer, direction, downstream_layer=refusal_layer):
    direction = direction / t.linalg.vector_norm(direction)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[-1]

    upstream_resid = None
    downstream_metric = None

    def upstream_hook(module, hook_args, output):
        nonlocal upstream_resid
        hidden = output[0] if isinstance(output, tuple) else output
        upstream_resid = hidden

    def downstream_hook(module, hook_args, output):
        nonlocal downstream_metric
        hidden = output[0] if isinstance(output, tuple) else output
        downstream_metric = hidden[:, -1, :].mul(direction.to(hidden)).sum()

    upstream_handle = core_model.layers[upstream_layer].register_forward_hook(upstream_hook)
    downstream_handle = core_model.layers[downstream_layer].register_forward_hook(downstream_hook)

    try:
        with t.enable_grad():
            outputs = model(**inputs)
            del outputs
    finally:
        upstream_handle.remove()
        downstream_handle.remove()

    if upstream_resid is None or downstream_metric is None:
        raise RuntimeError("Failed to capture upstream residuals or downstream metric.")

    grad = t.autograd.grad(downstream_metric, upstream_resid, retain_graph=False)[0]
    return grad[:, :prompt_len, :]



In [ ]:
## prompt templated
prompt_templated, prompt_templated_tokens = get_chat_template(PROMPT, tokenizer)

## get gradient for each SAE latent and sum over all positions
gradient = get_gradient(model, prompt_templated, UPSTREAM_LAYER, refusal_direction).float()
gradient_normalized = gradient / t.norm(gradient, p=2)
decoder = gemma2_sae.W_dec.detach().float()
sae_dec_gradient_sum = einops.einsum(decoder, gradient.squeeze(), 
                                   "d_sae d_model, ctx d_model -> d_sae")

## create a dataframe with the projection for each SAE latent
df_SAE_gradients = pd.DataFrame(sae_dec_gradient_sum.cpu().tolist(), columns = ["projection"])
df_SAE_gradients["Cosine Similarity"] = einops.einsum(decoder, gradient_normalized.squeeze(), 
                                                      "d_sae d_model, ctx d_model -> d_sae").cpu().tolist()
df_SAE_gradients = df_SAE_gradients.sort_values("projection").reset_index()
df_SAE_gradients["index"] = df_SAE_gradients["index"].astype(int)



In [ ]:
prompt_templated


## 2.1: Generate Histogram, and plot the top and bottom 3 SAE Latents

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
import numpy as np

metric = "Cosine Similarity"

# Slightly narrower figure with reduced height
plt.figure(figsize=(10, 5), dpi=300)

# Sorting the DataFrame and getting the top 3 and bottom 3 rows based on the "projection" column
top_3 = df_SAE_gradients.nlargest(3, metric)
bottom_3 = df_SAE_gradients.nsmallest(3, metric)

# Define gradient hues for top (blue) and bottom (red) lines
def generate_gradient_colors(start_color, end_color, n):
    return [mcolors.to_hex(c) for c in mcolors.LinearSegmentedColormap.from_list("", [start_color, end_color])(np.linspace(1, 0, n))]

top_colors = generate_gradient_colors("lightblue", "blue", 3)
bottom_colors = generate_gradient_colors("lightcoral", "red", 3)

# Plot the histogram with more bins for higher resolution
sns.histplot(data=df_SAE_gradients, x=metric, bins=50)

# Add vertical lines for the top 3 values
for color, (_, row) in zip(top_colors, top_3.iterrows()):
    plt.axvline(x=row[metric], color=color, linestyle=':', ymin=0, ymax=0.75)  # Vertical line to middle

# Add vertical lines for the bottom 3 values
for color, (_, row) in zip(bottom_colors, bottom_3.iterrows()):
    plt.axvline(x=row[metric], color=color, linestyle=':', ymin=0, ymax=0.75)  # Vertical line to middle

# Create legend handles for the top 3 and bottom 3
top_handles = [Line2D([0], [0], color=color, linestyle='-', label=f'{int(row["index"])}') 
               for color, (_, row) in zip(top_colors, top_3.iterrows())]
bottom_handles = [Line2D([0], [0], color=color, linestyle='-', label=f'{int(row["index"])}') 
                  for color, (_, row) in zip(bottom_colors, bottom_3.iterrows())]

# Add title with larger font size
plt.title("Cosine Similarity Between SAE Latents and Gemma Gradient", fontsize=18, pad=20)

# Add the first legend (Top 3)
legend1 = plt.legend(handles=top_handles, title="Top 3 Indices", loc='upper right', bbox_to_anchor=(1, 1), framealpha=1.0, fontsize=14, title_fontsize=16)
plt.gca().add_artist(legend1)  # Add the first legend to the plot

# Add the second legend (Bottom 3)
plt.legend(handles=bottom_handles, title="Bottom 3 Indices", loc='upper left', framealpha=1.0, fontsize=14, title_fontsize=16)

# Remove top and right spines
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

# Add labels with larger font size
plt.xlabel(metric, fontsize=16)
plt.ylabel("Count", fontsize=16)

# Increase tick label sizes
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Adjust layout to prevent cutting off
plt.tight_layout()

# Show the plot
plt.show()



## 2.2: Generate Neuronpedia Dashboard for Top and Bottom n SAE Latents

In [ ]:
pd.set_option('display.max_colwidth', None)

## bottom n
n = 5
feature_list = []
feature_interp_list = []
for feature_idx in df_SAE_gradients.iloc[:n]["index"]:
    feature_list.append(feature_idx)
    feature_interp_list.append(get_auto_interp(layer = UPSTREAM_LAYER, feature_idx = feature_idx))
    plot_dashboard(feature_idx, layer = UPSTREAM_LAYER)

df = pd.DataFrame(zip(feature_list, feature_interp_list), columns = ["Feature Index", "Auto Interp"])
print(df)


In [ ]:
feature_list = []
feature_interp_list = []
for feature_idx in df_SAE_gradients.iloc[-1:-n-1:-1]["index"]:
    feature_list.append(feature_idx)
    feature_interp_list.append(get_auto_interp(layer = UPSTREAM_LAYER, feature_idx = feature_idx))
    plot_dashboard(feature_idx, layer = UPSTREAM_LAYER)

df = pd.DataFrame(zip(feature_list, feature_interp_list), columns = ["Feature Index", "Auto Interp"])
print(df)


# Step 3: Test Model Output with steering

In [ ]:
def get_max_act_approx(layer, feature_idx):
    url = f"https://www.neuronpedia.org/api/feature/{NEURONPEDIA_MODEL}/{neuronpedia_layer_name(layer)}/{feature_idx}"

    headers = {"X-Api-Key": NEURONPEDIA_API_KEY}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return response.json()['maxActApprox']

class PromptOnlySteeringHook:
    def __init__(self, layer_idx: int, vector: t.Tensor, coeff: float, prompt_len: int):
        self.layer_idx = layer_idx
        self.vector = vector
        self.coeff = coeff
        self.prompt_len = prompt_len

    def __call__(self, module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        hidden = hidden.clone()
        steer_len = min(hidden.shape[1], self.prompt_len)
        hidden[:, :steer_len, :] += self.coeff * self.vector.view(1, 1, -1)
        if isinstance(output, tuple):
            return (hidden, *output[1:])
        return hidden

def generate_with_steering(
    model,
    upstream_layer: int,
    prompt: str,
    direction: Float[Tensor, "d_model"],
    steering_coefficient: float = None,
    max_new_tokens: int = 50,
    return_response_only: bool = True,
    steer_prompt_only: bool = True,
):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[-1]
    hook = PromptOnlySteeringHook(
        layer_idx=upstream_layer,
        vector=(direction / t.norm(direction, p=2)).to(model.device, dtype=next(model.parameters()).dtype),
        coeff=steering_coefficient,
        prompt_len=prompt_len,
    )
    hook_handle = core_model.layers[upstream_layer].register_forward_hook(hook)
    try:
        with t.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        hook_handle.remove()

    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    output = tokenizer.decode(new_ids, skip_special_tokens=True)
    return output.strip() if return_response_only else output

def explore_steering(
    model,
    direction: Float[Tensor, "d_model"],
    prompt: str,
    upstream_layer: int,
    steering_factors: list[float] = [0, 0.5, 1, 1.5],
    max_new_tokens: int = 75,
    direction_name: str = "",
    return_response_only: bool = True,
    steer_prompt_only: bool = True,
    max_act_approx: float = None,
) -> None:
    table = Table(show_header=False, show_lines=True, title=f"Output After Steering Along {direction_name}")

    for coef in tqdm(steering_factors):
        table.add_row(
            str(coef),
            generate_with_steering(
                model,
                upstream_layer,
                prompt,
                direction=direction,
                steering_coefficient=coef * max_act_approx,
                max_new_tokens=max_new_tokens,
                return_response_only=return_response_only,
                steer_prompt_only=steer_prompt_only,
            ).replace("\n", "↵")
        )

    rprint(table)




In [ ]:
## top 3 Get Dashboard and Steering Results
## Steer by x times the max activation for each SAE Latent
n = 3
for feature_idx in df_SAE_gradients.iloc[-1:-n-1:-1]["index"]:
    plot_dashboard(feature_idx, layer = UPSTREAM_LAYER)
    max_act_approx = get_max_act_approx(layer = UPSTREAM_LAYER, feature_idx = feature_idx)
    direction = gemma2_sae.W_dec[feature_idx,:].detach().float()
    explore_steering(model, direction, prompt_templated, UPSTREAM_LAYER, direction_name = f"SAE Latent {feature_idx}", 
                     max_act_approx = max_act_approx, max_new_tokens = 100)



In [ ]:
## Steer with the gradient
gradient_sum =gradient.sum(dim = 1).squeeze()
explore_steering(model, gradient_sum, prompt_templated, UPSTREAM_LAYER, direction_name = f"Gemma Gradient", max_act_approx = 30,
                 max_new_tokens = 100)



In [ ]:
## Steer with the decoder vector direction
explore_steering(model, refusal_direction, prompt_templated, UPSTREAM_LAYER, direction_name = DIRECTION_FEATURE_LABEL, max_act_approx = 30,
                 max_new_tokens = 100)



# Step 4: Generate Refusal Projection Plot after Steering

In [ ]:
## get refusal projection after steering along SAE latent by some coefficient

def get_projection_for_coefficient(
    model,
    upstream_direction: Float[Tensor, "d_model"],
    upstream_layer: int,
    prompt: str,
    downstream_direction: Float[Tensor, "d_model"],
    downstream_layer: int,
    steering_coefficient: float = None,
) -> float:
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    downstream_direction_normalized = downstream_direction / t.norm(downstream_direction, p=2)
    projection_holder = []

    def capture_projection(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        activation_pos = hidden[:, -1, :].squeeze().detach()
        projection_holder.append(t.dot(downstream_direction_normalized.to(activation_pos), activation_pos).item())

    hook = PromptOnlySteeringHook(
        layer_idx=upstream_layer,
        vector=(upstream_direction / t.norm(upstream_direction, p=2)).to(model.device, dtype=next(model.parameters()).dtype),
        coeff=steering_coefficient,
        prompt_len=inputs["input_ids"].shape[-1],
    )
    steer_handle = core_model.layers[upstream_layer].register_forward_hook(hook)
    capture_handle = core_model.layers[downstream_layer].register_forward_hook(capture_projection)

    try:
        with t.inference_mode():
            _ = model(**inputs)
    finally:
        steer_handle.remove()
        capture_handle.remove()

    return projection_holder[0]



## get projection after steering for top 3 SAE Latents and the Refusal Gradient

In [ ]:
steering_min = -5
steering_max = 50
## top 3
n = 3
n_random = 10


projections = []
for coef in range(steering_min, steering_max):
    projection = get_projection_for_coefficient(model, gradient_sum, UPSTREAM_LAYER, prompt_templated, refusal_direction, refusal_layer, steering_coefficient=coef)
    projections.append(projection)
df = pd.DataFrame(zip(range(steering_min, steering_max), projections), columns = ["Steering Coefficient", "Direction Projection"])
df["Perturbation Direction"] = "Gemma Gradient"

for feature_idx in df_SAE_gradients.iloc[-1:-n-1:-1]["index"]:
    direction = gemma2_sae.W_dec[feature_idx,:].detach().float()
    projections = []
    for coef in range(steering_min, steering_max):
        projection = get_projection_for_coefficient(model, direction, UPSTREAM_LAYER, prompt_templated, refusal_direction, refusal_layer, steering_coefficient=coef)
        projections.append(projection)
    df_new = pd.DataFrame(zip(range(steering_min, steering_max), projections), columns = ["Steering Coefficient", "Direction Projection"])
    df_new["Perturbation Direction"] = f"SAE Latent {feature_idx}"

    df = pd.concat([df, df_new])

first = True
random_index_list = random.sample(list(df_SAE_gradients["index"]), n_random)
for feature_idx in random_index_list:
    direction = gemma2_sae.W_dec[feature_idx,:].detach().float()
    projections = []
    for coef in range(steering_min, steering_max):
        projection = get_projection_for_coefficient(model, direction, UPSTREAM_LAYER, prompt_templated, refusal_direction, refusal_layer, steering_coefficient=coef)
        projections.append(projection)
    
    df_new = pd.DataFrame(zip(range(steering_min, steering_max), projections), columns = ["Steering Coefficient", "Direction Projection"])
    df_new["Perturbation Direction"] = f"Random SAE Latent {feature_idx}"

    if first:
        df_random = df_new.copy()
        first = False
    else:
        df_random = pd.concat([df_random, df_new])



In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Get overall min and max y values across both datasets
all_projections = pd.concat([df["Direction Projection"], df_random["Direction Projection"]])
y_min = all_projections.min() - 5
y_max = all_projections.max() + 5

sns.lineplot(data=df, x="Steering Coefficient", y="Direction Projection", hue="Perturbation Direction", ax=ax1)
starting_projection = df[df["Steering Coefficient"] == 0].iloc[0]["Direction Projection"]
ax1.axhline(y=starting_projection, color='black', linestyle='--')
ax1.set_ylim(y_min, y_max)
ax1.set_title("Gemma Gradient and Top SAE Latents")

sns.lineplot(data=df_random, x="Steering Coefficient", y="Direction Projection", hue="Perturbation Direction", ax=ax2, legend=False)
ax2.axhline(y=starting_projection, color='black', linestyle='--')
ax2.set_ylim(y_min, y_max)
ax2.set_title("Random SAE Latents")
